# 🏦 Credit Assessment Environment - GRPO Training

This notebook trains an LLM to make accurate loan underwriting decisions using **Group Relative Policy Optimization (GRPO)** from HuggingFace TRL.

## What the Agent Learns
- Follow RBI guidelines (CIBIL score, FOIR, LTV ratios)
- Detect **trap cases** (perfect profile with one hidden flaw)
- Make correct decisions: `approve`, `reject`, `request_docs`, `counter_offer`

## Theme Alignment
- **Theme #4: Self-Improvement (Primary)** — Performance-gated curriculum + Adversarial self-play + Self-generated challenges
- **Theme #3: World Modeling** — Real professional banking task with RBI regulatory constraints

## Training Modes (v3 — improved pipeline)

| Mode | Sections | Time (A100) | When to use |
|------|----------|-------------|-------------|
| Quick GRPO only | 7 | ~30 min | Smoke test, debugging |
| Curriculum only | 7b → 7c | ~60-90 min | If you can't spare SFT time |
| **Full pipeline (recommended)** | **7a → 7b → 7c → 8b** | **~2-2.5 hr** | **Real submission** |

### Recommended path

1. **Section 7a — SFT warmup** (~30 min). Supervised fine-tune on `(applicant, gold response)` pairs to teach the JSON-with-CoT format and the rule-walk reasoning style. Without this, GRPO struggles to discover correct behaviour for the harder loan types from sparse rewards.
2. **Re-run Section 5** so the trainer picks up the SFT-warmed adapter as the starting policy.
3. **Section 7b — Per-task curriculum** (~45 min). Personal → Vehicle → Home, with 20% replay from prior phases. Each phase pushes its best adapter to the HF Hub immediately so a Colab disconnect can't wipe out hours of work.
4. **Section 7c — Adversarial self-play** (~30 min). 2 rounds of targeted training on the strategies the model fails most.
5. **Section 7d — Push final adapter** (safety snapshot before the long eval).
6. **Section 8b — Fair eval with Wilson confidence intervals** (~20 min). The numbers that go into the slides and README come from here, *not* from the in-notebook eval, because this is the only path that scores baseline and trained on identical applicants with identical parsing.

---

**Runtime:** A100 GPU (Colab Pro)
**Model:** Qwen/Qwen2.5-7B-Instruct (auto-detected from GPU memory in Section 1)
**Expected Improvement:** ~60% (baseline) → 80%+ (after SFT + curriculum + adversarial)

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q trl>=0.17.0 transformers>=4.50.0 datasets accelerate peft bitsandbytes
!pip install -q matplotlib pydantic

In [ ]:
# Clone the Credit Assessment Environment.
# If the directory already exists (e.g. Colab runtime reused), `git pull` so
# we pick up the latest HF Space commit without a full reclone.
import os

if not os.path.exists("credit-assessment-env"):
    !git clone https://huggingface.co/spaces/iamnijin/credit-assessment-env
else:
    !cd credit-assessment-env && git pull --ff-only

%cd credit-assessment-env
!git log -1 --oneline  # Sanity check: show the commit this run is training on

In [ ]:
# Verify GPU and pick a model size that fits available memory.
#
# Sizing cheat sheet (with LoRA + bf16 + grad checkpointing):
#   T4  (16 GB)       -> Qwen2.5-1.5B-Instruct  (safe, ~45-60 min/phase)
#   L4  (24 GB)       -> Qwen2.5-3B-Instruct    (good balance)
#   A100 (40 GB)      -> Qwen2.5-7B-Instruct    (the default in README)
#   H100 (80 GB)      -> Qwen2.5-7B-Instruct
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_mem_gb:.1f} GB")

    # Auto-select model based on available memory. Override later in Section 5
    # by setting MODEL_NAME explicitly if you want a different size.
    if gpu_mem_gb >= 35:
        RECOMMENDED_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    elif gpu_mem_gb >= 22:
        RECOMMENDED_MODEL = "Qwen/Qwen2.5-3B-Instruct"
    else:
        RECOMMENDED_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    print(f"\nRecommended model for this GPU: {RECOMMENDED_MODEL}")
    print("(You can override this in Section 5 by setting MODEL_NAME manually.)")
else:
    print("⚠ No GPU detected — training will be unusable. Switch runtime to GPU.")
    RECOMMENDED_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

## 2. Environment Setup

The Credit Assessment Environment simulates Indian loan underwriting with:
- **3 difficulty levels**: Personal (Easy), Vehicle (Medium), Home (Hard)
- **Trap cases**: Profiles designed to trick pattern-matching LLMs
- **Asymmetric rewards**: Approving bad loans costs 3× more than rejecting good ones

In [ ]:
# Import standalone training utilities (avoids complex import chains)
import json
import random
from typing import Any

from train_utils import (
    DIFFICULTY_PROFILES,
    generate_applicant,
    calculate_ground_truth,
    calculate_reward,
    build_profile_text,
    CreditAssessmentAction,
    LoanDecision,
    generate_adversarial_case,
    AdversarialTracker,
    ADVERSARIAL_STRATEGIES,
)

# Test the environment
applicant = generate_applicant(task_id=1)  # Personal Loan
ground_truth = calculate_ground_truth(applicant)
profile = build_profile_text(applicant)

print("=" * 60)
print("Sample Loan Application")
print("=" * 60)
print(profile)
print(f"\nGround Truth Decision: {ground_truth}")

## 3. Dataset Generation

We generate synthetic loan applications with known ground truth decisions.
The dataset includes good, bad, borderline, and **trap** profiles.

In [ ]:
# v3: Single source of truth. Pull SYSTEM_PROMPT (now CoT + few-shot) and
# generate_dataset directly from train_grpo.py instead of redefining here.
# This guarantees the notebook always uses the same prompt/parser/reward
# logic as the standalone training script.
from datasets import Dataset
from train_grpo import SYSTEM_PROMPT, generate_dataset

# Eval pool of 200 keeps per-task numbers stable.
train_dataset = generate_dataset(300, seed=42)
eval_dataset = generate_dataset(200, seed=123)

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

from collections import Counter
gt_dist = Counter([s["ground_truth"] for s in train_dataset])
print(f"\nGround truth distribution: {dict(gt_dist)}")
print(f"\nSystem prompt now uses chain-of-thought + 2 few-shot examples.")
print(f"  Prompt length: {len(SYSTEM_PROMPT)} chars")

## 4. Reward Function

The reward function evaluates model decisions against ground truth:

| Scenario | Reward |
|----------|--------|
| Correct decision | +10 |
| Request docs when incomplete | +2 |
| Counter-offer catches high LTV | +5 |
| Approve bad loan | -15 |
| Reject good loan | -5 |
| Approve non-RERA home | -20 |

In [ ]:
# v3: Use the same reward + format functions as train_grpo.py. Both go
# through the shared lenient_parser, so chain-of-thought responses (CoT
# preamble + ```json answer block) get full credit instead of being scored
# as parse failures.
from train_grpo import (
    credit_assessment_reward,
    format_reward_score,
    combined_reward,
)

# Backward-compat alias for any cell that still calls the old name.
def format_reward(completions: list, **kwargs) -> list[float]:
    return [format_reward_score(c) for c in completions]

print("Reward functions imported from train_grpo (lenient parser, CoT-aware)")

## 5. Model & Training Configuration

We use:
- **Qwen2.5-7B-Instruct**: Strong reasoning, reliable JSON generation, fits A100 with LoRA
- **LoRA**: Parameter-efficient fine-tuning (trains only ~1% of parameters)
- **GRPO**: Group Relative Policy Optimization for reward maximization

**Stability fixes applied (v2):**
- `learning_rate=5e-6` — prevents policy overshooting on hard adversarial cases
- `max_grad_norm=0.5` — raised from 0.1: 0.1 was crushing gradient updates on our tiny effective batch (32 rollouts/step). Previous runs showed negative training losses, the signature of near-zero updates under aggressive clipping
- `beta=0.2` — raised from 0.1: stronger KL anchor prevents drift across curriculum → adversarial phase boundaries
- `num_generations=4` — balanced between advantage estimate quality and training speed; 8 was too slow on A100 with 400 samples/phase

In [ ]:
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
import torch

# Model selection (auto-detected in Section 1, override here if needed).
MODEL_NAME = globals().get("RECOMMENDED_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
print(f"Training model: {MODEL_NAME}")

# If sft_warmup ran first this points at the SFT-warmed adapter, otherwise
# GRPO starts from the raw base model. The warmup gives GRPO a much better
# starting policy (the previous run stayed at 0% on employment_trap_home
# because that decision was simply not in the base model's rollout
# distribution; SFT puts it there).
SFT_INIT_DIR = globals().get("SFT_INIT_DIR", "./grpo_credit_assessment_sft")
import os as _os
USE_SFT_INIT = _os.path.isdir(SFT_INIT_DIR)
print(f"SFT init: {'YES (' + SFT_INIT_DIR + ')' if USE_SFT_INIT else 'NO (cold-start GRPO)'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# v3 LoRA: doubled rank/alpha. With r=16 the adapter saturates after Phase 1
# and can't keep absorbing the new LTV/RERA logic in Phases 2 and 3.
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# v3 GRPO config:
#   * learning_rate dropped 5e-6 -> 1e-6 (loss was collapsing in <50 steps)
#   * beta 0.2 -> 0.3 (stronger reference anchor, less drift between phases)
#   * num_generations 4 -> 8 (lower-variance advantage estimates)
#   * max_completion_length 256 -> 512 (room for chain-of-thought + JSON)
training_args = GRPOConfig(
    output_dir="./grpo_credit_assessment",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-6,
    max_grad_norm=0.5,
    beta=0.3,
    num_generations=8,
    max_completion_length=512,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    report_to="none",
)

print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"GRPO generations per prompt: {training_args.num_generations}")
print(f"LoRA rank: {peft_config.r}  alpha: {peft_config.lora_alpha}")
print(f"Learning rate: {training_args.learning_rate}  beta (KL coef): {training_args.beta}")

In [ ]:
# Create GRPO Trainer.
#
# If SFT warmup ran (Section 7a), USE_SFT_INIT is True and we load the
# SFT-warmed adapter on top of the base model and pass it directly. We
# disable peft_config in that case because the adapter already exists and
# we want GRPO to refine it, not stack a fresh adapter on top.
#
# Otherwise we cold-start: pass the model name as a string and let
# GRPOTrainer attach a fresh LoRA via peft_config.
from transformers import AutoModelForCausalLM

if USE_SFT_INIT:
    print(f"Loading base model + SFT adapter from {SFT_INIT_DIR}...")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if training_args.bf16 else torch.float16,
        device_map="auto",
    )
    from peft import PeftModel
    sft_model = PeftModel.from_pretrained(base, SFT_INIT_DIR, is_trainable=True)
    print(f"  Trainable params from SFT adapter: ", end="")
    sft_model.print_trainable_parameters()

    trainer = GRPOTrainer(
        model=sft_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        peft_config=None,
        reward_funcs=combined_reward,
    )
    print("Trainer created with SFT-warmed adapter as starting policy.")
else:
    trainer = GRPOTrainer(
        model=MODEL_NAME,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
        reward_funcs=combined_reward,
    )
    print("Trainer created with cold-start policy (no SFT warmup).")

## 6. Pre-Training Evaluation (Baseline)

Let's measure the model's accuracy **before** training.

In [ ]:
# v3: use the shared lenient parser so CoT-preamble-then-```json responses
# are scored correctly. The old strict parser would silently mark every
# CoT response as a parse error and tank the baseline number.
from lenient_parser import parse_decision

def evaluate_model(model, tokenizer, dataset, num_samples=20, max_new_tokens=512):
    """Evaluate model accuracy on loan decisions.

    Uses greedy decoding (do_sample=False) so measurements are deterministic --
    critical when the threshold-gated curriculum depends on stable numbers.
    max_new_tokens bumped to 512 so chain-of-thought + JSON answer fits.
    """
    correct = 0
    total = 0
    results = []
    was_training = model.training
    model.eval()

    for i, sample in enumerate(dataset):
        if i >= num_samples:
            break

        prompt = tokenizer.apply_chat_template(
            sample["prompt"],
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        response = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        decision = parse_decision(response)
        is_correct = decision is not None and decision == sample["ground_truth"]
        results.append({
            "task_id": sample["task_id"],
            "ground_truth": sample["ground_truth"],
            "predicted": decision if decision is not None else "[PARSE_ERROR]",
            "correct": is_correct,
        })
        if is_correct:
            correct += 1
        total += 1

    if was_training:
        model.train()
    accuracy = correct / total if total > 0 else 0
    return accuracy, results


# ---------------------------------------------------------------------------
# OPTION A: Quick sanity check on the BASE model (no trainer needed)
# ---------------------------------------------------------------------------
# Run this FIRST, before SFT, to verify the new CoT system prompt is helping.
# We load the base model directly so this works even if you haven't yet
# created the GRPOTrainer in Section 5.
#
# What you're looking for:
#   * Old run baseline (terse prompt, strict parser):  ~50% overall
#   * New run baseline (CoT prompt,  lenient parser):  60-70% expected
# If you see 60%+ here, the prompt change alone is helping and SFT will
# stack on top. If you see <50%, something is wrong -- ping before SFT.
import torch
from transformers import AutoModelForCausalLM

print("Loading base model for baseline sanity check...")
_baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)

print("\nEvaluating BASE model with new CoT prompt (60 samples)...")
baseline_acc, baseline_results = evaluate_model(
    _baseline_model, tokenizer, eval_dataset, num_samples=60
)
print(f"\nBaseline (CoT prompt, lenient parser): {baseline_acc*100:.1f}%")

# Per-task breakdown so you can see where the prompt is helping most.
from collections import Counter
TASK_NAMES = {1: "personal", 2: "vehicle", 3: "home"}
by_task = {1: [0,0], 2: [0,0], 3: [0,0]}
for r in baseline_results:
    by_task[r["task_id"]][1] += 1
    if r["correct"]:
        by_task[r["task_id"]][0] += 1
print("\nPer-task baseline accuracy:")
for tid, name in TASK_NAMES.items():
    c, t = by_task[tid]
    pct = 100 * c / t if t else 0
    print(f"  {name:<10} {c:>2}/{t:<2}  ({pct:.0f}%)")

# Show one sample response so you can eyeball whether CoT is firing.
print("\nSample base-model response (first applicant):")
print("-" * 60)
prompt = tokenizer.apply_chat_template(
    eval_dataset[0]["prompt"], tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(_baseline_model.device)
with torch.no_grad():
    out = _baseline_model.generate(**inputs, max_new_tokens=512, do_sample=False,
                                   pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
print("-" * 60)
print(f"Ground truth: {eval_dataset[0]['ground_truth']}")

# Free the baseline model -- subsequent cells will reload it inside the trainer.
import gc
del _baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nBaseline sanity check complete. Memory freed.")
print("If accuracy is >= 60% with sane CoT-style responses, proceed to Section 7a (SFT).")
print("If accuracy < 50% or responses look broken, STOP and debug before burning credits.")

## 7. Training! 🚀

Now we train the model with GRPO. You'll see:
- **reward/mean**: Average reward (should increase)
- **loss**: Training loss (should decrease)
- **clip_ratio**: GRPO clipping ratio

In [ ]:
# Train!
print("Starting GRPO training...")
print("This will take ~30-60 minutes on a T4 GPU.")
print("="*60)

trainer.train()

print("="*60)
print("✓ Training complete!")

In [ ]:
# Save the model
trainer.save_model("./grpo_credit_assessment_final")
print("✓ Model saved to ./grpo_credit_assessment_final")

## 7a. SFT Warmup (Strongly Recommended)

Run supervised fine-tuning on `(applicant, gold response)` pairs **before** GRPO. This gives GRPO a working policy to refine instead of asking it to discover correct behaviour from sparse 4-class rewards.

The previous run stayed at 0% on `employment_trap_home` even after adversarial rounds — the correct decision was simply not in the base model's rollout distribution. SFT puts it there. Takes ~25-35 min on A100.

The cell saves a LoRA adapter to `./grpo_credit_assessment_sft/`. The Section 5 GRPO config above auto-detects it via `SFT_INIT_DIR` and uses it as the starting policy.

If you want to skip SFT (e.g. for a fast smoke test): just don't run this cell — the variable `USE_SFT_INIT` from Section 5 will fall back to `False` and GRPO will cold-start.

In [ ]:
# Free GPU memory before SFT — the trainer object from Section 5 is heavy.
import gc, torch
for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Run SFT warmup. ~25-35 min on A100 with 600 samples x 2 epochs.
import subprocess, sys
SFT_OUTPUT = "./grpo_credit_assessment_sft"
SFT_NUM_SAMPLES = 600
SFT_NUM_EPOCHS = 2

cmd = [
    sys.executable, "sft_warmup.py",
    "--model-name", MODEL_NAME,
    "--num-samples", str(SFT_NUM_SAMPLES),
    "--num-epochs", str(SFT_NUM_EPOCHS),
    "--output-dir", SFT_OUTPUT,
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f"SFT warmup failed with exit code {result.returncode}")

# Make sure subsequent cells see the SFT init.
import os
SFT_INIT_DIR = SFT_OUTPUT
USE_SFT_INIT = os.path.isdir(SFT_INIT_DIR)
print(f"\nSFT warmup complete. SFT_INIT_DIR = {SFT_INIT_DIR}")
print(f"USE_SFT_INIT = {USE_SFT_INIT}")
print(f"\n>>> NOTE: re-run Section 5 (Model & Training Configuration) so the trainer "
      f"picks up the SFT-warmed adapter, then continue with curriculum training.")

## 7b. Performance-Gated Curriculum Learning (Recommended)

**Run this INSTEAD of Section 7 for better results.**

Curriculum learning trains in 3 phases with mastery gating:
1. **Phase 1 (Easy)**: Only obvious good/bad cases — must reach **60% accuracy** to advance
2. **Phase 2 (Medium)**: Borderline cases — must reach **60% accuracy** to advance
3. **Phase 3 (Hard)**: All trap cases — no gate (nowhere left to advance)

The model **earns** advancement rather than advancing on a fixed timer. If accuracy stays below the threshold, the phase repeats **once** with fresh samples.

**v2 tuning:**
- `samples_per_phase: 400` (was 100) — enough gradient updates to actually learn
- `per-attempt eval samples: 50` (was 10) — 10 samples gave ±30% CI on the gate decision
- `mastery threshold: 60%` (was 65%) — realistic bar given the per-attempt eval precision
- `max_retries: 1` (was 2) — more retries tend to overfit on noise; we keep the best adapter per phase
- **Best-adapter-per-phase**: each retry saves its adapter if it beat the previous attempt. The best adapter is preserved on disk.

This aligns with **Theme #4: Self-Improvement** — recursive skill amplification through competence-gated progression.

In [ ]:
import os

# v3 curriculum:
#   * Per-task progression (Personal -> Vehicle -> Home) instead of difficulty
#     bands. Each phase introduces ONE new skill (Phase 2 adds vehicle-LTV,
#     Phase 3 adds home-tier-LTV + RERA) so the GRPO advantage signal isn't
#     mixing up "harder cases" with "new task".
#   * Replay buffer: each phase mixes 20% samples from prior phases so we
#     don't catastrophically forget personal-loan logic when learning home.
#   * Per-phase HF Hub push: every phase uploads the best adapter immediately
#     so a Colab disconnect can't wipe out hours of work (root cause of the
#     overnight loss when Safari put the tab to sleep).

MASTERY_THRESHOLD = 0.60
MAX_PHASE_RETRIES = 1
SAMPLES_PER_PHASE = 400
PHASE_EVAL_SAMPLES = 50
REPLAY_FRACTION = 0.2
BEST_ADAPTER_DIR = "./grpo_phase_best_adapter"
HUB_REPO_BASE = os.environ.get("HUB_REPO_BASE", "iamnijin/credit-assessment-curriculum")
PUSH_AFTER_EACH_PHASE = bool(os.environ.get("HF_TOKEN"))

CURRICULUM_PHASES = [
    ("personal", "Phase 1: Personal Loans (Foundation)",        MASTERY_THRESHOLD),
    ("vehicle",  "Phase 2: + Vehicle Loans (Adds LTV cap)",     MASTERY_THRESHOLD),
    ("home",     "Phase 3: + Home Loans (Tiered LTV + RERA)",   0.0),
]


def _filtered_dataset(loan_type, n, seed):
    """Generate `n` samples of a single loan_type. Oversamples then filters
    because generate_dataset cycles task_id by i%3."""
    pool = list(generate_dataset(n * 4, seed=seed, difficulty="all"))
    matching = [s for s in pool if s["loan_type"] == loan_type]
    return matching[:n]


def _build_phase_train(phase_idx, attempt):
    """Phase samples + replay from earlier phases."""
    seed = 42 + phase_idx * 100 + attempt
    n_total = SAMPLES_PER_PHASE
    n_replay = int(round(n_total * REPLAY_FRACTION)) if phase_idx > 0 else 0
    n_current = n_total - n_replay
    target_loan = CURRICULUM_PHASES[phase_idx][0]
    current = _filtered_dataset(target_loan, n_current, seed)
    replay = []
    if n_replay > 0:
        per_prior = max(1, n_replay // phase_idx)
        for prior in range(phase_idx):
            replay.extend(_filtered_dataset(
                CURRICULUM_PHASES[prior][0], per_prior, seed + 1000 + prior
            ))
    combined = current + replay
    random.Random(seed).shuffle(combined)
    return Dataset.from_list(combined), n_current, len(replay)


phase_results = []
phase_results_detail = []

for phase_idx, (target_loan, phase_name, threshold) in enumerate(CURRICULUM_PHASES):
    print(f"\n{'='*60}")
    print(f"{phase_name}")
    print(f"{'='*60}")

    # Eval pool for this phase: same loan type, larger pool for stable measurement.
    eval_dataset_phase = Dataset.from_list(
        _filtered_dataset(target_loan, PHASE_EVAL_SAMPLES * 2, seed=123 + phase_idx)
    )

    phase_acc = 0.0
    best_attempt_acc = -1.0
    best_attempt_idx = -1

    for attempt in range(MAX_PHASE_RETRIES + 1):
        if attempt > 0:
            print(f"\n  [Retry {attempt}/{MAX_PHASE_RETRIES}] "
                  f"Best so far: {best_attempt_acc*100:.1f}% — below {threshold*100:.0f}% threshold")

        phase_train, n_cur, n_rep = _build_phase_train(phase_idx, attempt)
        replay_str = f"  |  Replay: {n_rep}/{n_cur+n_rep}" if n_rep > 0 else ""
        print(f"  Samples: {len(phase_train)}  |  Loan type: {target_loan}{replay_str}")

        trainer.train_dataset = phase_train
        trainer.train()

        phase_acc, _ = evaluate_model(trainer.model, tokenizer, eval_dataset_phase, num_samples=PHASE_EVAL_SAMPLES)
        print(f"  Accuracy: {phase_acc*100:.1f}%"
              + (f" (threshold: {threshold*100:.0f}%)" if threshold > 0 else " (no gate on final phase)"))

        if phase_acc > best_attempt_acc:
            best_attempt_acc = phase_acc
            best_attempt_idx = attempt
            try:
                trainer.save_model(BEST_ADAPTER_DIR)
                print(f"  -> New best-in-phase adapter saved ({phase_acc*100:.1f}%)")
            except Exception as e:
                print(f"  WARN: could not save best adapter: {e}")

        if threshold == 0.0 or phase_acc >= threshold:
            if threshold > 0:
                print(f"  Mastery achieved -- advancing.")
            break
    else:
        print(f"  Threshold not reached after {MAX_PHASE_RETRIES+1} attempts -- advancing anyway.")

    if phase_acc < best_attempt_acc - 0.05 and os.path.isdir(BEST_ADAPTER_DIR):
        print(f"  WARN: final attempt regressed; best adapter preserved at {BEST_ADAPTER_DIR}")

    phase_results.append((phase_name, best_attempt_acc if best_attempt_acc >= 0 else phase_acc))
    phase_results_detail.append({
        "phase": phase_name,
        "loan_type": target_loan,
        "best_accuracy": best_attempt_acc if best_attempt_acc >= 0 else phase_acc,
        "final_attempt_accuracy": phase_acc,
        "attempts": best_attempt_idx + 1,
    })

    # Safety push: upload this phase's best adapter to HF Hub immediately.
    # If Colab disconnects after Phase 1 we don't lose the work.
    if PUSH_AFTER_EACH_PHASE:
        repo_id = f"{HUB_REPO_BASE}-phase{phase_idx+1}-{target_loan}"
        try:
            print(f"  Pushing phase adapter to HF Hub: {repo_id} ...")
            trainer.model.push_to_hub(repo_id, private=False)
            tokenizer.push_to_hub(repo_id, private=False)
            print(f"  Adapter live at https://huggingface.co/{repo_id}")
        except Exception as e:
            print(f"  WARN: push to hub failed ({e}); adapter still on disk at {BEST_ADAPTER_DIR}")

print("\n" + "="*60)
print("CURRICULUM RESULTS")
print("="*60)
for name, acc in phase_results:
    print(f"  {name}: {acc*100:.1f}%")

trainer.save_model("./grpo_curriculum_model")
print("\nCurriculum-trained model saved at ./grpo_curriculum_model")

## 7c. Adversarial Self-Play + Self-Generation (Run After Curriculum)

**Run this AFTER Section 7b for the full recursive self-improvement pipeline.**

Each round:
1. **Evaluate** on adversarial cases — record per-strategy failure rates
2. **Identify** the weakest strategy via `AdversarialTracker`
3. **Mix in** self-generated cases from the previous round (capped at 30%)
4. **Train** on targeted adversarial + hard cases
5. **Self-generate**: prompt the trained model to design trap cases targeting its own weaknesses — carry into next round

The self-generation loop is what makes this recursive: the model's own failure patterns shape what it trains on next.

In [ ]:
import torch
from datasets import Dataset

SELF_GEN_PROMPT = """You are a loan underwriting trainer designing test cases.

Design ONE loan application that looks approvable at first glance but contains exactly one hidden flaw.

The model currently struggles most with:
{weaknesses}

Output ONLY a JSON object — no explanation, no markdown:
{{
    "loan_type": "personal" | "vehicle" | "home",
    "credit_score": <integer 650-850>,
    "monthly_income": <integer 50000-1000000>,
    "foir": <float 0.15-0.60>,
    "employment_years": <integer 0-20>,
    "loan_amount": <integer 100000-20000000>,
    "documents_complete": true | false,
    "collateral_value": <integer, required for vehicle/home>,
    "ltv_ratio": <float 0.5-0.95, required for vehicle/home>,
    "rera_registered": true | false (required for home),
    "has_co_applicant": true | false,
    "trap_type": "<one line: what is the hidden flaw>"
}}"""


def generate_adversarial_dataset(num_samples, seed=42, tracker=None, target_weakness=True):
    random.seed(seed)
    samples = []
    if tracker and target_weakness:
        cases = tracker.generate_targeted_batch(num_samples, target_weakness=True)
        for case in cases:
            applicant = case["applicant"]
            strategy = case["strategy"]
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            samples.append({
                "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": profile_text}],
                "ground_truth": ground_truth,
                "task_id": {"personal": 1, "vehicle": 2, "home": 3}[applicant["loan_type"]],
                "applicant_data": json.dumps(applicant),
                "adversarial_strategy": strategy,
            })
    else:
        for i in range(num_samples):
            strategy = random.choice(ADVERSARIAL_STRATEGIES)
            applicant = generate_adversarial_case(strategy)
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            samples.append({
                "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": profile_text}],
                "ground_truth": ground_truth,
                "task_id": {"personal": 1, "vehicle": 2, "home": 3}[applicant["loan_type"]],
                "applicant_data": json.dumps(applicant),
                "adversarial_strategy": strategy,
            })
    return Dataset.from_list(samples)


def evaluate_adversarial_strategies(model, tokenizer, tracker, num_per_strategy=5):
    # v2: num_per_strategy default bumped from 2 → 5. At 2 samples per strategy,
    # "0%" and "100%" were mostly noise. 5 gives 20% resolution per strategy.
    results = {s: {"correct": 0, "total": 0} for s in ADVERSARIAL_STRATEGIES}
    model.eval()
    for strategy in ADVERSARIAL_STRATEGIES:
        for _ in range(num_per_strategy):
            applicant = generate_adversarial_case(strategy)
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            prompt = [{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": profile_text}]
            formatted = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                         pad_token_id=tokenizer.pad_token_id)
            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            try:
                if "```json" in response: response = response.split("```json")[1].split("```")[0]
                elif "```" in response: response = response.split("```")[1].split("```")[0]
                decision = json.loads(response.strip()).get("decision", "").lower()
                results[strategy]["total"] += 1
                if decision == ground_truth:
                    results[strategy]["correct"] += 1
                    tracker.record_result(strategy, True)
                else:
                    tracker.record_result(strategy, False)
            except:
                results[strategy]["total"] += 1
                tracker.record_result(strategy, False)
    model.train()
    return results


def self_generate_adversarial_cases(model, tokenizer, tracker, num_cases=15):
    """Prompt the trained model to design its own hard cases targeting its weaknesses."""
    weakness_summary = tracker.get_summary()
    top_weaknesses = sorted(weakness_summary.items(), key=lambda x: -x[1].get("failures", 0))[:3]
    weakness_str = "\n".join(f"- {s}: {d['failures']} failures" for s, d in top_weaknesses) \
                   if top_weaknesses else "- No data yet — generate varied trap cases"

    prompt_text = SELF_GEN_PROMPT.format(weaknesses=weakness_str)
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    samples = []
    attempts = 0
    max_attempts = num_cases * 4

    model.eval()
    with torch.no_grad():
        while len(samples) < num_cases and attempts < max_attempts:
            attempts += 1
            try:
                outputs = model.generate(**inputs, max_new_tokens=300, do_sample=True,
                                          temperature=0.9, pad_token_id=tokenizer.pad_token_id)
                response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:],
                                            skip_special_tokens=True).strip()
                if "```json" in response: response = response.split("```json")[1].split("```")[0]
                elif "```" in response: response = response.split("```")[1].split("```")[0]
                case = json.loads(response.strip())

                required = ["loan_type", "credit_score", "monthly_income", "foir",
                            "employment_years", "loan_amount", "documents_complete"]
                if not all(k in case for k in required): continue
                if case["loan_type"] not in ("personal", "vehicle", "home"): continue

                if case["loan_type"] in ("vehicle", "home") and "collateral_value" not in case:
                    case["collateral_value"] = int(case["loan_amount"] * 1.3)
                    case["ltv_ratio"] = case["loan_amount"] / case["collateral_value"]
                if case["loan_type"] == "home":
                    case.setdefault("rera_registered", True)
                    case.setdefault("has_co_applicant", False)
                    case.setdefault("property_type", "apartment")
                if case["loan_type"] == "vehicle":
                    case.setdefault("vehicle_type", "sedan")

                gt = calculate_ground_truth(case)
                if gt is None: continue

                profile = build_profile_text(case)
                samples.append({
                    "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                               {"role": "user", "content": profile}],
                    "ground_truth": gt,
                    "task_id": {"personal": 1, "vehicle": 2, "home": 3}[case["loan_type"]],
                    "applicant_data": json.dumps(case),
                    "adversarial_strategy": case.get("trap_type", "self_generated"),
                })
            except Exception:
                continue
    model.train()
    return samples



def evaluate_by_loan_type(model, tokenizer, num_samples_per_type=30):
    """Evaluate accuracy per loan type to detect catastrophic forgetting.

    v2 fix: previous version generated `num_samples_per_type` samples ONCE
    per loan_type and then filtered — meaning only ~1/3 of samples matched
    the target type, so each type got ~7 evaluated samples instead of 20.
    Now we pool `num_samples_per_type * 3` samples once (generate_dataset
    cycles task_id across 1,2,3 so each loan_type gets exactly the target
    count) and filter once.
    """
    results = {}
    model.eval()
    pool = generate_dataset(num_samples_per_type * 3, seed=999, difficulty="all")
    for loan_type in ["personal", "vehicle", "home"]:
        correct = 0
        total = 0
        for sample in pool:
            try:
                applicant_data = json.loads(sample.get("applicant_data", "{}"))
            except:
                applicant_data = {}
            if applicant_data.get("loan_type") != loan_type:
                continue
            prompt_text = tokenizer.apply_chat_template(
                sample["prompt"], tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=256, do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )
            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            try:
                if "```json" in response: response = response.split("```json")[1].split("```")[0]
                elif "```" in response: response = response.split("```")[1].split("```")[0]
                decision = json.loads(response.strip()).get("decision", "").lower()
                if decision == sample["ground_truth"]:
                    correct += 1
                total += 1
            except:
                total += 1
        results[loan_type] = {"correct": correct, "total": total,
                              "accuracy": correct / total if total > 0 else 0.0}
    model.train()
    print(f"    Personal: {results['personal']['correct']}/{results['personal']['total']} "
          f"({results['personal']['accuracy']*100:.0f}%)  |  "
          f"Vehicle: {results['vehicle']['correct']}/{results['vehicle']['total']} "
          f"({results['vehicle']['accuracy']*100:.0f}%)  |  "
          f"Home: {results['home']['correct']}/{results['home']['total']} "
          f"({results['home']['accuracy']*100:.0f}%)")
    return results

# ── Adversarial Self-Play Training ──────────────────────────────────────────

# v2 tuning:
#   - samples_per_round: 50 → 150 (3× more coverage of targeted strategy per round)
#   - adversarial_rounds: 3 → 2 (Round 3 produced no new signal in prior runs)
#   - num_per_strategy eval: 2 → 5 (2 was too noisy for targeting decisions)
#   - Snapshot curriculum checkpoint before we start — if adversarial regresses
#     overall accuracy, we can pitch the curriculum-only model instead.
tracker = AdversarialTracker()
adversarial_rounds = 2
samples_per_round = 150
num_per_strategy_eval = 5
self_gen_carry = []  # Self-generated cases from previous round

# Snapshot curriculum-end state as fallback
CURRICULUM_SNAPSHOT = "./grpo_curriculum_end_snapshot"
try:
    trainer.save_model(CURRICULUM_SNAPSHOT)
    print(f"📦 Curriculum snapshot saved to {CURRICULUM_SNAPSHOT}")
    print("   (Fallback if adversarial training regresses overall accuracy.)")
except Exception as e:
    print(f"⚠ Could not save curriculum snapshot: {e}")

# Baseline accuracy going INTO adversarial (for regression check)
print("\nMeasuring curriculum-only accuracy on eval set (for regression check)...")
pre_adversarial_acc, _ = evaluate_model(trainer.model, tokenizer, eval_dataset, num_samples=60)
print(f"  Curriculum-only accuracy: {pre_adversarial_acc*100:.1f}%")

print("\n" + "=" * 60)
print("ADVERSARIAL SELF-PLAY + SELF-GENERATION TRAINING")
print("=" * 60)

round_results = []
round_results_detail = []  # richer per-round data for training_log.json

for round_idx in range(adversarial_rounds):
    print(f"\n{'='*60}")
    print(f"ROUND {round_idx + 1}/{adversarial_rounds}")
    print(f"{'='*60}")

    # Step 1: Evaluate and update tracker
    print(f"\n  [Before Round {round_idx + 1}] Accuracy by loan type:")
    pre_round = evaluate_by_loan_type(trainer.model, tokenizer)
    print("Evaluating on adversarial strategies...")
    eval_results = evaluate_adversarial_strategies(trainer.model, tokenizer, tracker, num_per_strategy=num_per_strategy_eval)

    print("\nStrategy accuracy:")
    for strategy, results in sorted(eval_results.items(), key=lambda x: x[1]["correct"] / max(1, x[1]["total"])):
        acc = results["correct"] / max(1, results["total"]) * 100
        print(f"  {strategy}: {acc:.0f}%")

    weakness = tracker.get_weakness()
    print(f"\n→ Targeting weakness: {weakness}")

    # Step 2: Rule-based targeted dataset
    adversarial_data = generate_adversarial_dataset(samples_per_round, seed=42+round_idx+100,
                                                     tracker=tracker, target_weakness=True)
    adversarial_samples = list(adversarial_data)

    # Step 3: Mix in self-generated cases from previous round (30% cap)
    if self_gen_carry:
        max_carry = max(1, len(adversarial_samples) * 3 // 10)
        carry_to_use = self_gen_carry[:max_carry]
        adversarial_samples = carry_to_use + adversarial_samples
        print(f"  Mixed in {len(carry_to_use)} self-generated cases from previous round")

    # CRITICAL: Replay data from all difficulties to prevent catastrophic forgetting
    easy_replay = generate_dataset(samples_per_round // 4, seed=42+round_idx+200, difficulty="easy")
    medium_replay = generate_dataset(samples_per_round // 4, seed=42+round_idx+300, difficulty="medium")
    hard_replay = generate_dataset(samples_per_round // 4, seed=42+round_idx+400, difficulty="hard")
    
    replay_samples = list(easy_replay) + list(medium_replay) + list(hard_replay)
    combined = adversarial_samples + replay_samples
    random.shuffle(combined)
    train_data = Dataset.from_list(combined)
    
    print(f"  Data mix: {len(adversarial_samples)} adversarial + {len(replay_samples)} replay")

    # Step 4: Train — use 1 epoch per adversarial round to prevent overfitting
    # on the narrow targeted distribution (key stability fix).
    trainer.train_dataset = train_data
    prev_epochs = trainer.args.num_train_epochs
    trainer.args.num_train_epochs = 1
    print(f"\nTraining on {len(train_data)} samples (1 epoch)...")
    trainer.train()
    trainer.args.num_train_epochs = prev_epochs

    # Step 5: Measure improvement (POST-training)
    print(f"\n  [After Round {round_idx + 1}] Accuracy by loan type:")
    post_round = evaluate_by_loan_type(trainer.model, tokenizer)
    for lt in ["personal", "vehicle", "home"]:
        delta = post_round[lt]["accuracy"] - pre_round[lt]["accuracy"]
        arrow = "✅" if delta >= 0 else "❌"
        print(f"    {lt}: {delta*100:+.0f}% {arrow}")

    # Re-evaluate adversarial strategies AFTER training to get the true round accuracy.
    # (The pre-round eval_results only reflect the state going INTO this round.)
    print(f"\n  [After Round {round_idx + 1}] Re-evaluating adversarial strategies...")
    post_eval = evaluate_adversarial_strategies(trainer.model, tokenizer, tracker, num_per_strategy=num_per_strategy_eval)
    total_correct = sum(r["correct"] for r in post_eval.values())
    total = sum(r["total"] for r in post_eval.values())
    round_acc = total_correct / total if total > 0 else 0

    # Step 6: Self-generate hard cases for next round
    # v2: self-gen pool bumped from 15 → 30 (we still cap at 30% of the
    # adversarial mix during training, so extras provide a buffer against
    # invalid JSON generations).
    print(f"\nSelf-generating hard cases for next round...")
    self_gen_carry = self_generate_adversarial_cases(trainer.model, tokenizer, tracker, num_cases=30)
    print(f"Generated {len(self_gen_carry)} valid self-generated cases")

    round_results.append((round_idx + 1, weakness, round_acc, len(self_gen_carry)))

    # Detailed record consumed by Section 9c (training_log.json emission).
    # `eval_results` was the PRE-training eval for this round; `post_eval` is
    # POST-training for this round. Both are per-strategy dicts of {correct, total}.
    _pre_w = eval_results.get(weakness, {"correct": 0, "total": 1})
    _post_w = post_eval.get(weakness, {"correct": 0, "total": 1})
    if "round_results_detail" not in globals():
        round_results_detail = []
    round_results_detail.append({
        "round": round_idx + 1,
        "weakness": weakness,
        "round_acc": round_acc,
        "self_gen_count": len(self_gen_carry),
        "pre_targeted_correct": _pre_w.get("correct", 0),
        "pre_targeted_total": max(1, _pre_w.get("total", 1)),
        "post_targeted_correct": _post_w.get("correct", 0),
        "post_targeted_total": max(1, _post_w.get("total", 1)),
    })
    print(f"\nRound {round_idx + 1} complete: {round_acc*100:.1f}% accuracy")

print("\n" + "=" * 60)
print("ADVERSARIAL TRAINING SUMMARY")
print("=" * 60)
for round_num, weakness, acc, self_gen in round_results:
    print(f"  Round {round_num}: {acc*100:.1f}% | targeted: {weakness} | self-gen: {self_gen} cases")

print("\nFinal weakness analysis:")
for strategy, data in sorted(tracker.get_summary().items(), key=lambda x: x[1]["accuracy"]):
    print(f"  {strategy}: {data['accuracy']*100:.0f}% ({data['failures']} failures)")

trainer.save_model("./grpo_adversarial_final")
print("\n✓ Model saved to ./grpo_adversarial_final")

# ── Regression check: compare adversarial final vs curriculum snapshot ──
print("\n" + "=" * 60)
print("CURRICULUM vs CURRICULUM + ADVERSARIAL")
print("=" * 60)
post_adversarial_acc, _ = evaluate_model(trainer.model, tokenizer, eval_dataset, num_samples=60)
delta = post_adversarial_acc - pre_adversarial_acc
print(f"  Curriculum-only     : {pre_adversarial_acc*100:.1f}%")
print(f"  +Adversarial rounds : {post_adversarial_acc*100:.1f}%  (Δ {delta*100:+.1f}%)")

if delta < -0.03:
    print(f"\n  ⚠ Adversarial training REGRESSED overall accuracy by {abs(delta)*100:.1f}%.")
    print(f"     For the pitch, use the curriculum snapshot:")
    print(f"       {CURRICULUM_SNAPSHOT}")
    print(f"     Load it in a fresh trainer before running Section 8 post-training eval.")
elif delta > 0.03:
    print(f"\n  ✓ Adversarial training helped by {delta*100:.1f}%. Use ./grpo_adversarial_final.")
else:
    print(f"\n  ≈ Within noise (|Δ| < 3%). Either checkpoint is defensible for the pitch.")

## 7d. Push Final Adapter to HF Hub (Safety Snapshot)

Before running the long evaluation suite below, snapshot the trained adapter to the Hub. If anything later in the notebook crashes, fails to allocate memory, or the runtime disconnects, we still have the trained weights.

In [ ]:
import os

FINAL_REPO = os.environ.get("FINAL_HUB_REPO", "iamnijin/credit-assessment-curriculum")

if os.environ.get("HF_TOKEN"):
    print(f"Pushing final adapter to {FINAL_REPO} ...")
    try:
        trainer.model.push_to_hub(FINAL_REPO, private=False)
        tokenizer.push_to_hub(FINAL_REPO, private=False)
        print(f"Final adapter live at https://huggingface.co/{FINAL_REPO}")
    except Exception as e:
        print(f"Push failed: {e}")
        print(f"Adapter still on disk at ./grpo_curriculum_model -- you can push manually later.")
else:
    print("HF_TOKEN not set; skipping Hub push. Adapter on disk at ./grpo_curriculum_model")

## 8. Post-Training Evaluation

Let's measure improvement!

In [ ]:
# Post-training evaluation
# num_samples: 20 → 60 to match baseline eval. This is the number that lands
# in your hackathon chart, so it MUST be stable. 20 samples made "86% → 71% on
# Personal" look dramatic when it was literally 6/7 → 5/7 — 1 sample of noise.
print("Evaluating trained model (60 samples)...")
trained_acc, trained_results = evaluate_model(
    trainer.model, tokenizer, eval_dataset, num_samples=60
)

print(f"\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"📊 Baseline Accuracy: {baseline_acc*100:.1f}%")
print(f"📈 Trained Accuracy:  {trained_acc*100:.1f}%")
print(f"🚀 Improvement:       {(trained_acc - baseline_acc)*100:+.1f}%")
print("="*60)

In [ ]:
# Detailed breakdown by task
import pandas as pd

task_names = {1: "Personal Loan", 2: "Vehicle Loan", 3: "Home Loan"}

# Baseline breakdown
baseline_df = pd.DataFrame(baseline_results)
baseline_by_task = baseline_df.groupby("task_id")["correct"].mean()

# Trained breakdown
trained_df = pd.DataFrame(trained_results)
trained_by_task = trained_df.groupby("task_id")["correct"].mean()

print("\nAccuracy by Loan Type:")
print("-" * 50)
print(f"{'Task':<20} {'Baseline':>12} {'Trained':>12} {'Δ':>10}")
print("-" * 50)
for task_id in [1, 2, 3]:
    base = baseline_by_task.get(task_id, 0)
    trained = trained_by_task.get(task_id, 0)
    delta = trained - base
    print(f"{task_names[task_id]:<20} {base*100:>10.1f}% {trained*100:>10.1f}% {delta*100:>+9.1f}%")

## 8b. Fair Apples-to-Apples Evaluation (THIS is the proof)

The numbers in Section 8 above use the in-notebook `evaluate_model` path. Even though we now use the lenient parser there too, the baseline-vs-trained comparison reported by `train_grpo.py` historically used two slightly different sample pools.

`scripts/fair_eval.py` runs the **base model** and the **trained adapter** through the **identical** applicant set with the **identical** parser, then reports per-task accuracy with **95% Wilson confidence intervals** so we can tell which deltas are statistically meaningful vs sampling noise.

This is the chart that goes on Slide 6 and into the README.

In [ ]:
# Free GPU memory before re-loading both models for a fair comparison.
import gc, torch, subprocess, sys, os
for _name in ("trainer",):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Run fair_eval. Pulls the base model + the trained adapter (from local disk)
# and scores them on the *same* applicant pool with the *same* lenient parser.
# Outputs go to assets/fair_eval_results.json + assets/fair_eval_chart.png.
ADAPTER_PATH = "./grpo_curriculum_model"
NUM_FAIR_SAMPLES = 120  # ~30 minutes on A100 for a 7B model

cmd = [
    sys.executable, "scripts/fair_eval.py",
    "--base-model", MODEL_NAME,
    "--adapter-path", ADAPTER_PATH,
    "--num-samples", str(NUM_FAIR_SAMPLES),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    print(f"\nWARN: fair_eval exited with code {result.returncode}")
else:
    print("\nFair eval complete. Results in assets/fair_eval_results.json")
    print("Chart in assets/fair_eval_chart.png")
    if os.path.exists("assets/fair_eval_chart.png"):
        from IPython.display import Image, display
        display(Image("assets/fair_eval_chart.png"))

## 9b. Generate Hackathon-Ready Results & Narrative

Run this cell after training to generate:
1. **Before/After accuracy chart** (for your presentation)
2. **Trap case examples** (for your pitch)
3. **Copy-paste ready pitch summary**

In [ ]:
# ============================================================
# HACKATHON RESULTS GENERATOR
# ============================================================
# Uses your actual baseline_acc and trained_acc from above

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ------------------------------------------------------------
# 1. GENERATE RESULTS CHART
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overall accuracy comparison
categories = ['Baseline\n(Pre-training)', 'Trained\n(Post-GRPO)']
accuracies = [baseline_acc * 100, trained_acc * 100]
colors = ['#e74c3c', '#27ae60']

bars = axes[0].bar(categories, accuracies, color=colors, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Overall Loan Decision Accuracy', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].axhline(y=100, color='gray', linestyle='--', alpha=0.3, label='Perfect (Rule-Based)')

# Add value labels
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=14)

# Add improvement arrow
improvement = (trained_acc - baseline_acc) * 100
mid_y = (baseline_acc + trained_acc) / 2 * 100
axes[0].annotate(f'+{improvement:.1f}%', 
                 xy=(1, trained_acc * 100 - 5), 
                 fontsize=16, fontweight='bold', color='#27ae60',
                 ha='center')

# Right: By loan type breakdown
task_names = ['Personal\n(Easy)', 'Vehicle\n(Medium)', 'Home\n(Hard)']

# Calculate per-task accuracy from results
baseline_by_task = [0, 0, 0]
trained_by_task = [0, 0, 0]
baseline_counts = [0, 0, 0]
trained_counts = [0, 0, 0]

for r in baseline_results:
    idx = r['task_id'] - 1
    baseline_by_task[idx] += 1 if r['correct'] else 0
    baseline_counts[idx] += 1

for r in trained_results:
    idx = r['task_id'] - 1
    trained_by_task[idx] += 1 if r['correct'] else 0
    trained_counts[idx] += 1

baseline_by_task = [baseline_by_task[i]/max(1,baseline_counts[i])*100 for i in range(3)]
trained_by_task = [trained_by_task[i]/max(1,trained_counts[i])*100 for i in range(3)]

x = range(len(task_names))
width = 0.35

bars1 = axes[1].bar([i - width/2 for i in x], baseline_by_task, width, 
                    label='Baseline', color='#e74c3c', edgecolor='black')
bars2 = axes[1].bar([i + width/2 for i in x], trained_by_task, width,
                    label='Trained', color='#27ae60', edgecolor='black')

axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy by Loan Type (Difficulty)', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(task_names)
axes[1].set_ylim(0, 100)
axes[1].legend()

# Add value labels
for bar in bars1:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('hackathon_results.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✓ Chart saved to hackathon_results.png")
print("  (Right-click → Save Image to download)")

# ------------------------------------------------------------
# 2. TRAP CASE EXAMPLES (for your pitch)
# ------------------------------------------------------------

print("\n" + "="*60)
print("TRAP CASES FOR YOUR PITCH")
print("="*60)

# Generate example trap cases
trap_examples = [
    {
        "name": "The High-Income Trap",
        "profile": "₹6L/month income, 20% FOIR, 12 years experience",
        "hidden_flaw": "CIBIL score: 699 (just 1 point below 700)",
        "correct": "REJECT",
        "why_fails": "LLMs see high income and approve without checking CIBIL threshold"
    },
    {
        "name": "The Perfect-Profile Trap", 
        "profile": "CIBIL 820, ₹2L/month, 25% FOIR, 10 years",
        "hidden_flaw": "Property NOT RERA registered",
        "correct": "REJECT",
        "why_fails": "Everything screams 'approve' - RERA field is easy to overlook"
    },
    {
        "name": "The RBI Tier Trap",
        "profile": "₹1.2Cr home loan, LTV 78%",
        "hidden_flaw": "Loans >₹75L have max 75% LTV (RBI rule)",
        "correct": "COUNTER_OFFER",
        "why_fails": "Must know tiered limits: ≤30L→90%, 30-75L→80%, >75L→75%"
    }
]

for i, trap in enumerate(trap_examples, 1):
    print(f"\n### Trap {i}: {trap['name']}")
    print(f"Profile: {trap['profile']}")
    print(f"Hidden Flaw: {trap['hidden_flaw']}")
    print(f"Correct Decision: {trap['correct']}")
    print(f"Why LLMs Fail: {trap['why_fails']}")

# ------------------------------------------------------------
# 3. COPY-PASTE PITCH SUMMARY
# ------------------------------------------------------------

print("\n" + "="*60)
print("COPY-PASTE PITCH SUMMARY")
print("="*60)

print(f"""
## Credit Assessment Environment: Teaching LLMs to Be Loan Officers

**The Problem:** LLMs pattern-match. Banking requires precision. 
One overlooked criterion (CIBIL 699 vs 700) = ₹50L NPA.

**Our Solution:** Self-improving RL environment with:
- Trap cases targeting LLM weaknesses
- Curriculum learning (easy → medium → hard)  
- Adversarial self-play based on failure analysis
- Asymmetric rewards matching real banking risk

**Results:**
- Baseline: {baseline_acc*100:.1f}%
- Trained: {trained_acc*100:.1f}%
- Improvement: +{improvement:.1f}%

**Why It Matters:**
Every bank in India processes thousands of loans daily. 
An agent that catches edge cases = millions saved in NPAs.
""")

## 9. Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

# Get training history
history = trainer.state.log_history

# Extract metrics
steps = []
rewards = []
losses = []

for entry in history:
    if "reward" in entry:
        steps.append(entry.get("step", 0))
        rewards.append(entry["reward"])
    if "loss" in entry and entry.get("step", 0) > 0:
        if len(losses) < len(steps):
            losses.append(entry["loss"])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curve
axes[0].plot(steps[:len(rewards)], rewards, 'b-', linewidth=2, label='Average Reward')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Reward')
axes[0].set_title('🎯 Reward During Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
categories = ['Baseline', 'Trained']
accuracies = [baseline_acc * 100, trained_acc * 100]
colors = ['#ff6b6b', '#51cf66']
bars = axes[1].bar(categories, accuracies, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('📈 Before vs After Training')
axes[1].set_ylim(0, 100)

# Add value labels
for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved to training_results.png")

## 10. Try the Trained Model!

Let's test on a trap case to see if training helped.

In [ ]:
# Generate a trap case
random.seed(12345)
trap_applicant = generate_applicant(task_id=3)  # Home loan (hardest)
trap_gt = calculate_ground_truth(trap_applicant)
trap_profile = build_profile_text(trap_applicant)

print("=" * 60)
print("TEST CASE: Home Loan Application")
print("=" * 60)
print(trap_profile)
print(f"\n🎯 Correct Decision: {trap_gt}")
print("=" * 60)

# Get model's decision
test_prompt = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": trap_profile}
]

formatted = tokenizer.apply_chat_template(
    test_prompt, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(formatted, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("\n🤖 Model's Response:")
print(response)

# Check if correct
try:
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0]
    elif "```" in response:
        response = response.split("```")[1].split("```")[0]
    
    parsed = json.loads(response.strip())
    decision = parsed.get("decision", "")
    
    if decision == trap_gt:
        print("\n✅ CORRECT!")
    else:
        print(f"\n❌ Incorrect. Model said '{decision}', should be '{trap_gt}'")
except:
    print("\n⚠️ Could not parse response")

## 11. Push to HuggingFace Hub (Optional)

Upload your trained model to share with others!

In [ ]:
# Uncomment and run to push to hub
# from huggingface_hub import login
# login()  # Enter your HF token

# trainer.push_to_hub("your-username/credit-assessment-grpo")
# print("✓ Model pushed to HuggingFace Hub!")

---

## 12. Emit `training_log.json` + Generate Labeled Plots

This cell turns everything that's currently in memory (`baseline_results`, `trained_results`, `phase_results`, `round_results_detail`, `trainer.state.log_history`, `tracker`) into the JSON schema documented in `training_log.example.json` — then runs `scripts/generate_plots.py` to produce the four labeled plots the hackathon judge guide asks for, and zips the outputs for one-click download.

**What this produces:**
- `training_log.json` — the full machine-readable training record
- `assets/reward_curve.png`
- `assets/per_task_accuracy.png`
- `assets/adversarial_rounds.png`
- `assets/curriculum_phases.png`
- `training_outputs.zip` — everything above bundled for download

In [ ]:
import json
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

TASK_ID_TO_TYPE = {1: "personal", 2: "vehicle", 3: "home"}


def _per_task_accuracy(results):
    """results: list of {task_id, correct}. Returns {type: accuracy}."""
    by_task = {1: [0, 0], 2: [0, 0], 3: [0, 0]}  # [correct, total]
    for r in results:
        tid = r.get("task_id")
        if tid not in by_task:
            continue
        by_task[tid][1] += 1
        if r.get("correct"):
            by_task[tid][0] += 1
    return {
        TASK_ID_TO_TYPE[tid]: (c / t if t else 0.0)
        for tid, (c, t) in by_task.items()
    }


def _overall_accuracy(results):
    if not results:
        return 0.0
    return sum(1 for r in results if r.get("correct")) / len(results)


def _extract_reward_curve(log_history, max_points=60):
    """trainer.state.log_history entries contain 'step' + 'reward' (and maybe 'kl')."""
    points = []
    for entry in log_history:
        if "reward" not in entry:
            continue
        points.append({
            "step": int(entry.get("step", 0)),
            "reward": float(entry["reward"]),
            "kl": float(entry.get("kl", 0.0)) if "kl" in entry else None,
        })
    # Downsample if too many.
    if len(points) > max_points:
        stride = len(points) // max_points
        points = points[::stride]
    return points


gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

# Build the structured log. All variables below should already exist from
# prior cells — if you skipped a section (e.g. adversarial), the corresponding
# entries will just be empty lists, and generate_plots.py will skip that plot.
training_log = {
    "meta": {
        "model_name": MODEL_NAME,
        "mode": "curriculum+adversarial+self_generation",
        "seed": 42,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "hardware": gpu_name,
        "num_train_samples": len(train_dataset) if "train_dataset" in globals() else 0,
        "wall_clock_seconds": None,
    },
    "baseline": {
        "per_task": _per_task_accuracy(baseline_results) if "baseline_results" in globals() else {},
        "overall": _overall_accuracy(baseline_results) if "baseline_results" in globals() else 0.0,
        "mean_reward": None,
    },
    "trained": {
        "per_task": _per_task_accuracy(trained_results) if "trained_results" in globals() else {},
        "overall": _overall_accuracy(trained_results) if "trained_results" in globals() else 0.0,
        "mean_reward": None,
    },
    "reward_curve": _extract_reward_curve(trainer.state.log_history) if "trainer" in globals() else [],
    "curriculum": {
        "phase_mastery_threshold": MASTERY_THRESHOLD if "MASTERY_THRESHOLD" in globals() else 0.60,
        "phases": [
            {"name": name, "steps": [0, 0], "final_eval": float(acc), "retries": 0}
            for (name, acc) in (phase_results if "phase_results" in globals() else [])
        ],
    },
    "adversarial_rounds": [
        {
            "round": r["round"],
            "targeted_strategy": r["weakness"],
            "pre_round": {
                "targeted_accuracy": r["pre_targeted_correct"] / r["pre_targeted_total"]
            },
            "post_round": {
                "targeted_accuracy": r["post_targeted_correct"] / r["post_targeted_total"]
            },
            "self_generated_count": r["self_gen_count"],
        }
        for r in (round_results_detail if "round_results_detail" in globals() else [])
    ],
}

# Mean-reward backfill: take the last logged 'reward' as the trained mean.
curve = training_log["reward_curve"]
if curve:
    training_log["trained"]["mean_reward"] = curve[-1]["reward"]
    training_log["baseline"]["mean_reward"] = curve[0]["reward"]

log_path = Path("training_log.json")
log_path.write_text(json.dumps(training_log, indent=2))
print(f"✓ Wrote {log_path} ({log_path.stat().st_size:,} bytes)")
print(f"  Baseline overall: {training_log['baseline']['overall']*100:.1f}%")
print(f"  Trained  overall: {training_log['trained']['overall']*100:.1f}%")
print(f"  Curriculum phases: {len(training_log['curriculum']['phases'])}")
print(f"  Adversarial rounds: {len(training_log['adversarial_rounds'])}")
print(f"  Reward-curve points: {len(training_log['reward_curve'])}")

# Run the plot generator. If it's not present in the cloned repo (older HF
# Space commit), fall back to a pip-installable matplotlib regeneration.
Path("assets").mkdir(exist_ok=True)
if Path("scripts/generate_plots.py").exists():
    result = subprocess.run(
        ["python", "scripts/generate_plots.py", "training_log.json"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("⚠ generate_plots.py returned non-zero:", result.stderr)
else:
    print("⚠ scripts/generate_plots.py not found in the cloned Space.")
    print("  Make sure you pushed today's local changes to the `hf` remote")
    print("  before re-cloning. Plots will be regenerated manually below.")

# Zip everything relevant for a one-click download.
zip_path = "/content/training_outputs.zip"
plot_files = [str(p) for p in Path("assets").glob("*.png")]
fair_eval_files = [str(p) for p in Path("assets").glob("fair_eval_results*.json")]
files_to_zip = ["training_log.json"] + plot_files + fair_eval_files

# Use `zip` CLI — resilient if shutil is missing fields on Colab.
!rm -f {zip_path}
!zip -j {zip_path} {" ".join(files_to_zip)}

print(f"\n📦 Bundle ready: {zip_path}")
print("\nTo download:")
print("  from google.colab import files; files.download('/content/training_outputs.zip')")


In [ ]:
from google.colab import files
files.download('/content/training_outputs.zip')

---

## Summary

This notebook demonstrated:

1. **Environment**: Credit Assessment with 3 difficulty levels and trap cases
2. **Reward Model**: Asymmetric penalties matching real banking risk
3. **Training**: GRPO with LoRA for parameter-efficient fine-tuning
4. **Results**: Measurable improvement in loan decision accuracy

### Training Paths (all code is ready to run):

| Path | What to Run | Description |
|------|-------------|-------------|
| Quick | Section 7 | Standard training, all difficulties mixed |
| **Recommended** | Section 7b | Curriculum: Easy → Medium → Hard |
| Full Pipeline | 7b then 7c | Curriculum + Adversarial self-play |